# 10. Raw-data profile of the public datasets (DL-59, step 0)

**Question.** Before loading anything into the warehouse: what do the downloaded CMS, CDC, SEC and FDA files actually look like, and what rules does the ingestion need?

**Answer in one line.** The files are clean and consistent enough to ingest, with a short list of rules (suppression markers, placeholder rows, changing layouts, name variants, one bad date) that the loading code must handle; two items still need a decision or a small extra download.

**What this notebook did and did not look at.** It profiles *structure and quality only*: row counts, empty values, distinct names, ranges, duplicate keys, naming variants and coverage. It does **not** compute any share, trend or ranking, and per-year Zilretta counts were deliberately not written out (only whether each year has rows). The profile files are produced by `python -m oa_market_intelligence.external.profile_all`, which is tested, and are committed in `data/reference/external_profile/` so every number below can be traced.

> **Disclosure carried into the protocol.** Earlier, while checking downloads, we already saw a few outcome-like numbers (for example the count of provider rows billing Zilretta by year). The protocol labels the Medicare trend check (E2) accordingly and puts its weight on the tests written afterwards.

In [1]:
import json
from pathlib import Path

import pandas as pd

P = Path("../data/reference/external_profile")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)


def load(name):
    return json.loads((P / f"{name}.json").read_text(encoding="utf-8"))


print(sorted(p.name for p in P.iterdir()))

['asp_quarters.csv', 'cdc_places.json', 'geographic_variation.json', 'ma_geographic_variation.json', 'manifest_check.json', 'medicaid_sdud.json', 'nppes.json', 'openpayments.json', 'openpayments_coverage.json', 'openpayments_product_name_variants.csv', 'openpayments_reconciliation_2025.json', 'openpayments_zilretta_manufacturers.csv', 'partb_geo.json', 'partb_geo_code_descriptions.csv', 'partb_provider.json', 'partb_provider_specialty_names.csv', 'partd_candidate_drugs.csv', 'partd_geo.json', 'partd_provider.json', 'partd_provider_prescriber_types.csv', 'sec_filings.csv', 'specialty_crosswalk_candidates.csv']


## 1. Are the downloaded files complete and unchanged?

Every download was logged with its source link, size and SHA-256 checksum. The check re-reads each file and compares (checksums for files under 200 MB; size for the large ones).

In [2]:
m = load("manifest_check")
{k: (v if not isinstance(v, list) else len(v)) for k, v in m.items()}

{'entries': 147,
 'missing': 0,
 'size_mismatch': 0,
 'sha256_checked': 140,
 'sha256_mismatch': 0}

All 147 manifest entries exist, every size matches, and all 140 checksums that could be re-computed match. Nothing is missing or partial.

## 2. Medicare Part B by geography and by provider

Two views of the same billing data: state-level totals (2019 to 2024), and provider-level rows for the approved code list (filtered API pull for 2019 to 2023, plus 2024 filtered from the full file).

In [3]:
geo = load("partb_geo")
prov = load("partb_provider")
rows = []
for y in sorted(geo):
    g, p = geo[y], prov[y]
    rows.append({
        "year": y,
        "geo_rows": g["n_rows"], "geo_columns": g["n_columns"], "geo_duplicate_keys": g["duplicate_keys"],
        "min_patients_geo": g["columns"]["Tot_Benes"]["min"],
        "provider_rows": p["n_rows"], "provider_duplicate_keys": p["duplicate_keys"],
        "min_patients_provider": p["columns"]["Tot_Benes"]["min"],
        "specialty_names": p["columns"]["Rndrng_Prvdr_Type"]["n_distinct"],
        "org_rows": p["columns"]["Rndrng_Prvdr_Ent_Cd"]["top_values"].get("O", 0),
        "facility_rows": p["columns"]["Place_Of_Srvc"]["top_values"].get("F", 0),
    })
pd.DataFrame(rows).set_index("year")

,geo_rows,geo_columns,geo_duplicate_keys,min_patients_geo,provider_rows,provider_duplicate_keys,min_patients_provider,specialty_names,org_rows,facility_rows
year,,,,,,,,,,
2019,273211,15,0,11.0,185295,0,11.0,68,685,7969
2020,268149,15,0,11.0,156538,0,11.0,67,561,6450
2021,271635,15,0,11.0,163580,0,11.0,65,565,6958
2022,270673,15,0,11.0,164243,0,11.0,65,561,6783
2023,268634,15,0,11.0,162299,0,11.0,65,525,6619
2024,268350,15,0,11.0,164940,0,11.0,68,538,6650


- **Same 15 columns every year, no duplicate keys, nothing unparseable.** The smallest patient count anywhere is **11**: CMS hides any figure built from 10 or fewer patients, so small cells are *missing*, not zero. That biases small segments and is noted in the lineage.
- **About 0.3% of provider rows are organisations** (entity code O). They are excluded from "number of providers" counts.
- **Almost all billing is office-based** (setting O); facility rows are about 4%.

In [4]:
names = pd.read_csv(P / "partb_provider_specialty_names.csv")
pv = names.pivot_table(index="specialty", columns="year", values="n_rows_all_approved_codes", aggfunc="sum")
years_present = pv.notna().sum(axis=1)
print(f"{len(pv)} specialty names in total; {int((years_present == 6).sum())} appear in all six years; {int((years_present < 6).sum())} do not")
pv[years_present < 6].notna().astype(int)

76 specialty names in total; 61 appear in all six years; 15 do not


year,2019,2020,2021,2022,2023,2024
specialty,,,,,,
Certified Nurse Midwife,1,0,0,0,0,0
Chiropractic,0,0,0,0,1,1
Clinic or Group Practice,1,1,1,1,0,0
Clinical Laboratory,1,0,0,0,0,0
Colorectal Surgery (Proctology),1,1,0,0,0,1
Dentist,0,0,0,0,1,0
Micrographic Dermatologic Surgery,0,1,1,1,1,1
Neuropsychiatry,1,0,0,0,0,0
Optometry,0,0,0,0,0,1


The 61 names present in every year include all twelve Medicare names the specialty crosswalk in section 9 needs (checked). The 15 that come and go are very rare (for example Undersea and Hyperbaric Medicine, a dentist category) and are grouped with "other".

In [5]:
d = pd.read_csv(P / "partb_geo_code_descriptions.csv")
present = d.pivot_table(index="code", columns="year", values="description", aggfunc="count").fillna(0).astype(int)
changed = d.groupby("code").description.nunique()
print("codes whose wording changes between years:", list(changed[changed > 1].index))
present[(present == 0).any(axis=1)]

codes whose wording changes between years: ['20610', '20611', 'J7321']


year,2019,2020,2021,2022,2023,2024
code,,,,,,
J1010,0,0,0,0,0,1
J2919,0,0,0,0,0,1
J7331,0,1,1,1,1,1
J7332,0,1,1,1,1,1


**The code set changes over time.** J1010 and J2919 exist only from 2024; J7331 and J7332 only from 2020. The ingestion keeps a code-by-year validity table, treats methylprednisolone acetate as one drug family across J1010, J1020, J1030 and J1040, and never reads a missing year as "zero use". The wording of two procedure codes and one hyaluronic code changed between years; only the text differs.

## 3. Medicare Part D by geography (the NSAID and oral steroid side)

In [6]:
pdg = load("partd_geo")
pd.DataFrame([{ "year": y, "rows": v["n_rows"], "columns": v["n_columns"], "duplicate_keys": v["duplicate_keys"],
                "min_patients": v["columns"]["Tot_Benes"]["min"]} for y, v in sorted(pdg.items())]).set_index("year")

,rows,columns,duplicate_keys,min_patients
year,,,,
2019,114645,22,0,11.0
2020,115410,22,0,11.0
2021,114891,22,0,11.0
2022,115396,22,0,11.0
2023,115936,22,0,11.0
2024,117661,22,0,11.0


In [7]:
c = pd.read_csv(P / "partd_candidate_drugs.csv")
print(c.family.value_counts().to_dict(), "| names that do not appear in all six years:", int((c.years_present != "2019,2020,2021,2022,2023,2024").sum()), "of", len(c))
c[c.family == "nsaid"].generic_name.value_counts().head(10)

{'steroid': 75, 'nsaid': 73} | names that do not appear in all six years: 46 of 148


generic_name
Naproxen Sodium                  5
Ketorolac Tromethamine           4
Diclofenac Potassium             4
Diclofenac Sodium                4
Diclofenac Sodium/Misoprostol    3
Ibuprofen                        3
Celecoxib                        3
Diclofenac Epolamine             3
Indomethacin                     3
Meloxicam                        3
Name: count, dtype: int64

Layout and keys are stable across the six years. The name-matching nominated **73 NSAID and 75 oral-steroid name variants**; 46 of the 148 are not present every year (brand names come and go). The protocol appendix will fix the final drug list from this table by generic name, which is the stable key.

## 4. Quarterly price files

The files are Excel-style tables saved as CSV, with title lines above the header. The layout changed over the years.

In [8]:
a = pd.read_csv(P / "asp_quarters.csv")
print(len(a), "files;", a.quarter.nunique(), "distinct quarters from", a.quarter.min(), "to", a.quarter.max())
print("Zilretta's code present in:", int(a.approved_codes_present.fillna("").str.contains("J3304").sum()), "of", len(a), "files")
variants = a.groupby("header").agg(files=("file", "count"), first=("quarter", "min"), last=("quarter", "max"), columns=("n_columns", "first"))
variants.reset_index(drop=True)

26 files; 26 distinct quarters from 2019Q3 to 2025Q4
Zilretta's code present in: 26 of 26 files


,files,first,last,columns
0,5,2024Q1,2025Q4,11
1,4,2023Q2,2025Q1,12
2,2,2023Q4,2024Q3,13
3,13,2019Q4,2023Q1,10
4,2,2019Q3,2020Q4,12


Five different header layouts appear (columns added in 2023, trailing empty columns in some files), so the reader must find columns **by name, not position**, and locate the header row rather than assume it. The two 2019 and four 2020 files you downloaded by hand are the later "updated" releases of those quarters; that is recorded in the manifest.

## 5. Open Payments (industry payments to physicians)

Seven yearly files, 2019 to 2025, about 88 million rows in all.

In [9]:
op = load("openpayments")
rows = []
for y, v in sorted(op.items()):
    c = v["columns"]
    rows.append({
        "year": y, "rows": v["n_rows"], "columns": v["n_columns"],
        "first_date": c["Date_of_Payment"]["min"], "last_date": c["Date_of_Payment"]["max"],
        "unparseable_dates": c["Date_of_Payment"]["n_unparseable"],
        "negative_amounts": c["Total_Amount_of_Payment_USDollars"]["n_negative"],
        "zero_amounts": c["Total_Amount_of_Payment_USDollars"]["n_zero"],
        "min_payments_counted": c["Number_of_Payments_Included_in_Total_Amount"]["min"],
    })
pd.DataFrame(rows).set_index("year")

,rows,columns,first_date,last_date,unparseable_dates,negative_amounts,zero_amounts,min_payments_counted
year,,,,,,,,
2019,10558933,20,2019-01-01,2019-12-31,0,0,0,0.0
2020,5851438,20,2020-01-01,2020-12-31,0,0,0,0.0
2021,11558469,20,2021-01-01,2021-12-31,0,0,0,0.0
2022,13322266,20,2022-01-01,2022-12-31,0,0,0,1.0
2023,14734121,20,2023-01-01,2023-12-31,0,0,3,1.0
2024,15498687,20,0002-11-30,2024-12-31,0,0,0,1.0
2025,16131856,20,2025-01-01,2025-12-31,0,0,0,1.0


- **Identical 91-column layout every year.** Dates parse everywhere and amounts are never negative.
- **One bad date:** the 2024 file has a payment dated in the year **0002**. The rule: a payment must fall inside its program year, otherwise it is flagged and excluded from monthly series.
- **Zero "payments counted"** appears in some 2019 to 2021 rows and **three zero-dollar rows** in 2023: flagged, not dropped silently.
- **These are the 2026 republication of every year**, so older years include later corrections; they are not "as first published".

In [10]:
y24 = op["2024"]["columns"]
for k in ["Covered_Recipient_Type", "Change_Type", "Dispute_Status_for_Publication"]:
    print(k, y24[k]["top_values"])
print("recipients with no specialty (teaching hospitals):", y24["Covered_Recipient_Specialty_1"]["n_empty"])
print("distinct specialty strings:", y24["Covered_Recipient_Specialty_1"]["n_distinct"])

Covered_Recipient_Type {'Covered Recipient Physician': 9965643, 'Covered Recipient Non-Physician Practitioner': 5495656, 'Covered Recipient Teaching Hospital': 37388}
Change_Type {'UNCHANGED': 15074113, 'NEW': 273413, 'CHANGED': 151161}
Dispute_Status_for_Publication {'No': 15498360, 'Yes': 327}
recipients with no specialty (teaching hospitals): 37388
distinct specialty strings: 388


Recipients are physicians, non-physician practitioners (nurse practitioners and physician assistants) and teaching hospitals; hospitals have no specialty. For counting *physicians paid* the rule is physicians only, with practitioners as a sensitivity run.

In [11]:
v = pd.read_csv(P / "openpayments_product_name_variants.csv")
print("Zilretta spellings found:", v[v.product_name.str.contains("zilretta", case=False)].product_name.tolist())
print("paying entities for Zilretta:"); print(pd.read_csv(P / "openpayments_zilretta_manufacturers.csv")[["manufacturer"]].to_string(index=False))
print("reconciliation, 2025 Zilretta rows:", load("openpayments_reconciliation_2025")["zilretta_rows_seen"], "(an independent text search found 3,275)")
v[~v.product_name.str.contains("zilretta", case=False)].assign(years=lambda d: d.years_present.str.count(",") + 1)[["product_name", "years"]].head(40)

Zilretta spellings found: ['Zilretta']
paying entities for Zilretta:
                       manufacturer
         Flexion Therapeutics, Inc.
Pacira Pharmaceuticals Incorporated
          Pacira Therapeutics, Inc.
reconciliation, 2025 Zilretta rows: 3275 (an independent text search found 3,275)


,product_name,years
0,BIOLOGICS CONSUMABLES HYALURONIC ACID OTHER,1
1,DUROLANE,2
2,Durolane,5
3,EUFLEXXA,7
4,Euflexxa,1
5,GELSYN 3,2
6,GELSYN-3,5
7,GENVISC 850 SODIUM HYALURONATE,4
8,Gel-One Cross-linked Hyaluronate,4
9,Gelsyn,1


- **Zilretta has a single spelling** ("Zilretta") and appears in any of five product slots; the 2025 count reconciles exactly with an independent search (3,275).
- **Three different companies pay for it** (Flexion Therapeutics, and two Pacira entities), which matches the ownership change. So the filter is **the product name, never the payer**.
- **Hyaluronic-acid products appear under many spellings** (case, "Gel-One Cross-linked Hyaluronate", "Gelsyn 3" vs "Gelsyn-3"): they need a normalisation map. One false match (`Triamcinolone-Moxifloxacin PF`, an eye-surgery mix) and one generic bucket are excluded.
- **Kenalog never appears as a product name**, so promotion of the generic steroid comparator cannot be measured; only Zilretta and the hyaluronic products can.

## 6. Medicare Geographic Variation, Medicare Advantage and CDC PLACES

In [12]:
gv = load("geographic_variation")
print("rows", gv["n_rows"], "| duplicate keys", gv["duplicate_keys"], "| MA rate unparseable ('*' = suppressed):", gv["columns"]["MA_PRTCPTN_RATE"]["n_unparseable"], "| range", gv["columns"]["MA_PRTCPTN_RATE"]["min"], "to", gv["columns"]["MA_PRTCPTN_RATE"]["max"])
print("levels", gv["columns"]["BENE_GEO_LVL"]["top_values"], "| years", gv["columns"]["YEAR"]["top_values"])
pl = load("cdc_places")
print("PLACES rows", pl["n_rows"], "| years in file", pl["columns"]["Year"]["top_values"], "| value types", pl["columns"]["Data_Value_Type"]["top_values"])
print("empty values with footnote:", pl["columns"]["Data_Value"]["n_empty"], pl["columns"]["Data_Value_Footnote_Symbol"]["top_values"])

rows 36994 | duplicate keys 33 | MA rate unparseable ('*' = suppressed): 1240 | range 0.0 to 0.9755
levels {'County': 35146, 'State': 1815, 'National': 33} | years {'2022': 3365, '2023': 3365, '2024': 3365, '2021': 3364, '2015': 3363, '2014': 3362, '2016': 3362, '2017': 3362, '2018': 3362, '2019': 3362, '2020': 3362}
PLACES rows 229298 | years in file {'2023': 197848, '2022': 31450} | value types {'Crude prevalence': 114649, 'Age-adjusted prevalence': 114649}
empty values with footnote: 66 {'*': 66, '#': 14}


- **Geographic Variation:** the 33 "duplicate keys" are placeholder rows named "Territory" and "ZZ" with no geography code and `*` in every number. They are dropped. Elsewhere `*` means *suppressed* and becomes missing, never zero (1,240 Advantage rates are suppressed: 1,174 county rows and the 66 placeholder rows). State rows come in three age levels, county rows only in "all ages".
- **CDC PLACES:** the file mixes two data years (2022 and 2023) and two measure types (crude and age-adjusted prevalence), so the loader selects **one year and one type per measure**; 66 values are blank with a footnote.

## 7. Provider registry (NPPES)

In [13]:
n = load("nppes")
c = n["columns"]
print("rows", n["n_rows"], "| entity types", c["Entity Type Code"]["top_values"], "| blank entity type", c["Entity Type Code"]["n_empty"])
print("distinct 'state' values:", c["Provider Business Practice Location Address State Name"]["n_distinct"], "(should be about 60)")
print("distinct primary taxonomy codes:", c["Healthcare Provider Taxonomy Code_1"]["n_distinct"])
print("deactivation dates present on", n["n_rows"] - c["NPI Deactivation Date"]["n_empty"], "rows; snapshot last updated", c["Last Update Date"]["max"])

rows 9798758 | entity types {'1': 7471371, '2': 1972058} | blank entity type 355329
distinct 'state' values: 1080 (should be about 60)
distinct primary taxonomy codes: 873
deactivation dates present on 374443 rows; snapshot last updated 2026-09-14


- **The "state" field has 1,080 distinct values** (abbreviations, full names, typos, foreign addresses), so it must be cleaned to two-letter codes before use.
- **355,329 rows have no entity type, enumeration date or state**, which looks like stripped deactivated records; they are excluded.
- **It is one snapshot (September 2026), not history**, so provider counts can only be today's denominators, and are flagged as such.
- **Open item:** primary specialty comes as a taxonomy code (873 distinct). Mapping it to a specialty needs the public NUCC taxonomy code list, which is **not yet downloaded**.

## 8. Company filings

In [14]:
s = pd.read_csv(P / "sec_filings.csv")
print(len(s), "filings; period-end date read automatically for", int(s.period_end.notna().sum()))
s.groupby("company").agg(filings=("file", "count"), first_filed=("filed", "min"), last_filed=("filed", "max"),
                         zilretta_mentions=("n_zilretta_mentions", "sum"), viscosupplement_mentions=("n_viscosupplement_mentions", "sum"))

86 filings; period-end date read automatically for 60


,filings,first_filed,last_filed,zilretta_mentions,viscosupplement_mentions
company,,,,,
Anika Therapeutics,31,2019-02-27,2026-07-29,0,559
Bioventus,23,2021-03-26,2026-08-05,0,184
Flexion Therapeutics,12,2019-02-28,2021-11-12,3231,2
Pacira BioSciences,20,2021-11-03,2026-08-04,2138,0


All 86 filings are real documents with the expected products named: Zilretta in Flexion and Pacira, hyaluronic products in Anika and Bioventus. The automatic period-end reader works for 60 of 86; it is only a convenience, because the revenue figures are **transcribed by hand** with filing, accession number and page.

## 9. Specialty names: IQVIA against Medicare (draft for review)

The Medicare file names specialties differently from IQVIA. The code only *suggests* matches; every decision below is a person's, to be fixed in the protocol before any comparison is run.

In [15]:
x = pd.read_csv(P / "specialty_crosswalk_candidates.csv").fillna("")
print(len(x), "IQVIA specialty names;", int(x.exact_match.sum()), "match a Medicare name exactly after normalising")
proposed = pd.DataFrame([
    ("ANESTHESIOLOGY", "Anesthesiology", "exact; nurse anesthetists excluded"),
    ("FAMILY PRACTICE", "Family Practice", "exact"),
    ("INTERNAL MEDICINE", "Internal Medicine", "exact"),
    ("NURSE PRACTITIONER", "Nurse Practitioner", "exact"),
    ("ORTHOPEDIC SURGERY", "Orthopedic Surgery", "exact; IQVIA spine-surgery group is separate and excluded"),
    ("OSTEOPATHIC MEDICINE", "Osteopathic Manipulative Medicine", "weak: different definitions, flag in results"),
    ("PAIN MEDICINE", "Interventional Pain Management + Pain Management", "two Medicare names combined"),
    ("PHYSICAL MEDICINE & REHAB", "Physical Medicine and Rehabilitation", "name differs, same specialty"),
    ("PHYSICIAN ASSISTANT", "Physician Assistant", "exact"),
    ("RHEUMATOLOGY", "Rheumatology", "exact"),
    ("SPORTS MEDICINE", "Sports Medicine", "exact"),
], columns=["IQVIA group", "Medicare name(s)", "note"])
proposed

50

 IQVIA specialty names; 26 match a Medicare name exactly after normalising


,IQVIA group,Medicare name(s),note
0,ANESTHESIOLOGY,Anesthesiology,exact; nurse anesthetists excluded
1,FAMILY PRACTICE,Family Practice,exact
2,INTERNAL MEDICINE,Internal Medicine,exact
3,NURSE PRACTITIONER,Nurse Practitioner,exact
4,ORTHOPEDIC SURGERY,Orthopedic Surgery,exact; IQVIA spine-surgery group is separate and excluded
5,OSTEOPATHIC MEDICINE,Osteopathic Manipulative Medicine,"weak: different definitions, flag in results"
6,PAIN MEDICINE,Interventional Pain Management + Pain Management,two Medicare names combined
7,PHYSICAL MEDICINE & REHAB,Physical Medicine and Rehabilitation,"name differs, same specialty"
8,PHYSICIAN ASSISTANT,Physician Assistant,exact
9,RHEUMATOLOGY,Rheumatology,exact


Eleven of the twelve IQVIA groups used in the adoption model have a clear Medicare counterpart; the twelfth, "rare (grouped)", is excluded from the comparison. The weakest link is osteopathic medicine, and the protocol will say so rather than hide it.

## 10. Ingestion rules this profile produced

| # | Source | Rule the loader must follow |
|---|---|---|
| R1 | All files | Read everything as text; cast to types in Silver with explicit rules (codes keep leading zeros) |
| R2 | CMS Part B and Part D | Cells from 10 or fewer patients are suppressed by CMS: the minimum seen is 11. Absence means *unknown*, not zero |
| R3 | Geographic Variation | `*` means suppressed, load as NULL. Drop placeholder rows with no geography code ("Territory", "ZZ") |
| R4 | Part B provider | Exclude organisation rows (entity code O) from provider counts. Key (NPI, code, setting) is unique |
| R5 | Part B codes | Keep a code-by-year validity table (J1010 and J2919 from 2024; J7331 and J7332 from 2020). Methylprednisolone acetate is one drug family |
| R6 | Part B setting | Office versus facility are loaded separately; office is about 96% of rows |
| R7 | Price files | Find the header row and read columns by name (five layouts). Quarter comes from the file label |
| R8 | Open Payments | Identical 91 columns 2019 to 2025. Filter by product name in any of five slots, never by payer. Payment date must fall inside its program year (one 2024 row is dated year 0002). Flag zero-count and zero-dollar rows. Use physicians for "physicians paid"; practitioners as a sensitivity |
| R9 | Open Payments products | Normalise hyaluronic product spellings with an explicit map; exclude `Triamcinolone-Moxifloxacin PF`; Kenalog is not measurable |
| R10 | CDC PLACES | One data year and one value type per measure |
| R11 | Provider registry | Clean the state field to two letters; exclude blank-entity rows; treat it as a single 2026 snapshot |
| R12 | Company filings | Revenue transcribed by hand with filing, accession number and page; automatic period parsing is a convenience only |
| R13 | Reconciliation | After loading, row counts must equal these profiles (for example 2025 Zilretta rows in Open Payments = 3,275) |

### Still open before ingestion
1. **NUCC taxonomy list** (small public file) to turn registry taxonomy codes into specialties. Needs a download decision.
2. **The specialty crosswalk above** needs your approval; osteopathic medicine is the weak link.
3. **Final drug lists** for the NSAID and oral-steroid side, to be fixed from the candidate table.

## 11. Two optional files: profiled, not loaded

Part D by Provider and Drug (2024) and Medicaid State Drug Utilization (2024) were downloaded as optional context and no pre-registered rule uses them, so they were not profiled with the others. They are profiled here so that every file we hold has been looked at. **Structure and quality only**: no volumes or amounts by product, no trends, and no prescriber names or cities (the Part D file contains them; the profile counts their empties and never lists them). The profiles are in `data/reference/external_profile/` (`partd_provider.json`, `partd_provider_prescriber_types.csv`, `medicaid_sdud.json`).

In [16]:
medicaid = load("medicaid_sdud")["2024"]
partd = load("partd_provider")["2024"]
mc = medicaid["columns"]
pc = partd["columns"]
facts = pd.DataFrame(
    [
        ("rows", f"{medicaid['n_rows']:,}", f"{partd['n_rows']:,}"),
        ("years in the file", ", ".join(mc["Year"]["top_values"]), "2024"),
        ("distinct drug codes or names", f"{medicaid['n_distinct_ndc']:,} codes, {medicaid['n_distinct_product_names']:,} names", f"{partd['n_distinct_drug_names']:,} names"),
        ("distinct prescribers", "-", f"{partd['n_distinct_prescribers']:,}"),
        ("duplicate keys", medicaid["duplicate_keys"], partd["duplicate_keys"]),
        ("rows with suppressed (blank) volumes", f"{mc['Suppression Used']['top_values']['true']:,} ({mc['Suppression Used']['top_values']['true'] / medicaid['n_rows']:.1%})", "claims below 11 are not published (minimum " + f"{pc['Tot_Clms']['min']:.0f})"),
        ("drug codes that are not 11 digits", medicaid["n_ndc_not_eleven_digits"], "-"),
        ("negative values", "none" if mc["Total Amount Reimbursed"]["n_negative"] == 0 else "present", "none" if pc["Tot_Clms"]["n_negative"] == 0 else "present"),
    ],
    columns=["", "Medicaid drug utilisation 2024", "Part D by provider 2024"],
)
facts

,,Medicaid drug utilisation 2024,Part D by provider 2024
0,rows,"5,308,407","28,023,892"
1,years in the file,2024,2024
2,distinct drug codes or names,"50,574 codes, 15,097 names","3,159 names"
3,distinct prescribers,-,"1,139,455"
4,duplicate keys,0,0
5,rows with suppressed (blank) volumes,"2,666,743 (50.2%)",claims below 11 are not published (minimum 11)
6,drug codes that are not 11 digits,0,-
7,negative values,none,none


In [17]:
print("Medicaid: payment types", dict(mc["Utilization Type"]["top_values"]))
print("Medicaid: state codes", mc["State"]["n_distinct"], "- the code XX has", f"{mc['State']['top_values']['XX']:,}", "rows")
print("Medicaid: names matching the approved lists (names only, no volumes):")
for label, names in medicaid["matching_product_names"].items():
    print("  ", label, names)
print()
print("Part D: prescriber types", partd["n_prescriber_types"])
src = pc["Prscrbr_Type_Src"]["top_values"]
print("Part D: where the specialty comes from", {k: f"{v / partd['n_rows']:.1%}" for k, v in src.items()})
print("Part D: NSAID name pairs", partd["n_nsaid_name_pairs"], "| oral steroid name pairs", partd["n_oral_steroid_name_pairs"])
print("Part D: Zilretta name pairs (names only):", partd["zilretta_name_pairs"])
print("Part D: columns present but never listed:", partd["identifying_columns_not_listed"])

Medicaid: payment types {'MCOU': 2781645, 'FFSU': 2526762}
Medicaid: state codes 53 - the code XX has 320,216 rows
Medicaid: names matching the approved lists (names only, no volumes):
   zilretta ['ZILRETTA', 'ZILRETTA 3', 'ZILRETTA S']
   hyaluronic ['EUFLEXXA', 'EUFLEXXA 1', 'EUFLEXXA 2', 'MONOVISC', 'MONOVISC 8', 'ORTHOVISC', 'SYNVISC', 'SYNVISC ON', 'SYNVISC SY', 'SYNVISC-ON']
   steroid ['CELESTONE', 'CELESTONE-', 'KENALOG 80', 'KENALOG-10', 'KENALOG-40', 'KENALOG-80']

Part D: prescriber types 183
Part D: where the specialty comes from {'Claim-Specialty': '90.0%', 'NPPES-Specialty': '8.0%', 'NPPES-Taxonomy': '2.1%'}
Part D: NSAID name pairs 57 | oral steroid name pairs 47
Part D: Zilretta name pairs (names only): [['Zilretta', 'Triamcinolone Acetonide']]
Part D: columns present but never listed: ['Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City']


**What the profile shows.**

*Medicaid drug utilisation 2024.* One year only (four quarters), two payment types (fee-for-service and managed care) and 53 state codes, including an `XX` code with about 320,000 rows that would need a rule before any use. **About half the rows (50%) have their volumes suppressed** (blank, because the cell is below the publication minimum), so a thin product's cells are mostly hidden. Drug codes are all eleven digits and must be read as text to keep leading zeros; the numbers are zero-padded text (`00000000226.000`) and parse cleanly; there are no negatives and no duplicate keys. Product names are cut to ten characters (for example `KENALOG-40`, `SYNVISC ON`), so matching needs an explicit name list. Zilretta, hyaluronic and steroid names are all present (existence only: no volume was looked at, on purpose).

*Part D by provider and drug 2024.* 28 million rows, 1.1 million prescribers, about 3,200 drug names, no duplicate keys, 183 prescriber types. Claims below 11 are not published. The specialty of about 10% of rows comes from the provider registry and not from claims, so it is a different source of specialty. The file contains prescriber names and cities, which the profile never lists.

**A correction.** We had said Zilretta is not in Part D because it is a physician-administered injection normally billed under Part B. **It does appear in this Part D file by name** (`Zilretta`, `Triamcinolone Acetonide`), so at least some Zilretta is billed through Part D. We have not looked at how much, and we will not before a rule is written down. The Part D geography file we already load keeps only NSAID and oral steroid names, so it does not carry Zilretta.

**Status.** Both files are profiled, not loaded, and used by no rule. If either is used later it needs a protocol entry first; the Part D provider file would also be local-only (it is prescriber-level).